# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata object
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', None))
print("License:", metadata.license)
print("Published Date:", getattr(metadata, 'datePublished', None))
print("Authors (@id):", [author['@id'] for author in metadata.author])

## 2. Data Overview

Review available record sets and their fields, referenced by `@id`.

The Croissant schema describes record sets and their fields. We'll enumerate them by their `@id`.

In [ ]:
# List all record sets in the dataset with their @id, label, and available fields
record_sets = list(dataset.record_sets())

record_set_ids = []
for rs in record_sets:
    print("RecordSet @id:", rs['@id'])
    label = rs.get('label', rs.get('name', rs['@id']))
    print("  Name/Label:", label)
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Field @ids:", [field['@id'] for field in fields])
    print()
    record_set_ids.append(rs['@id'])

## 3. Data Extraction
Load records from each record set into pandas DataFrames, referencing each set by its `@id`.

In [ ]:
# Extract the data for each record set
dataframes = {}

for record_set_id in record_set_ids:
    # Load records from this record set
    records = list(dataset.records(record_set=record_set_id))
    # Create DataFrame
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record Set {record_set_id} Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)

Apply data processing, such as filtering, normalization, and grouping. Use fields referenced by their `@id`.

In [ ]:
# Example EDA: Filtering and Normalizing for mental health scores
# For demonstration, select a record set containing GAD-7, PHQ-9, PSQ, or a demographics field

# Select a record set with a numeric field (for example purposes, let's choose the first one with GAD-7 Score)
chosen_record_set = None
numeric_field_id = None

# Search for likely numeric fields (e.g., GAD7, PHQ9, PSQ)
for rs_id, df in dataframes.items():
    for col in df.columns:
        if "gad" in col.lower() or "phq" in col.lower() or "psq" in col.lower():
            chosen_record_set = rs_id
            numeric_field_id = col
            break
    if chosen_record_set:
        break

if chosen_record_set and numeric_field_id:
    print(f"Using Record Set {chosen_record_set} and Numeric Field {numeric_field_id}")
    threshold = 10
    filtered_df = dataframes[chosen_record_set][dataframes[chosen_record_set][numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a key demographic (e.g., 'age', 'gender', 'level_of_education')
    group_fields = [col for col in filtered_df.columns if any(field in col.lower() for field in ["age", "gender", "education", "village"])]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped average {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric mental health score field found in any record set!")

## 5. Visualization

Visualize data distributions or relationships between fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the chosen numeric field if EDA was successful
if chosen_record_set and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[chosen_record_set][numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} Scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouped by demographic, plot average scores
    if group_fields:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Average {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Average {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.
- Key data entities and fields were referenced by their Croissant `@id`, ensuring schema consistency.
- We loaded records, performed filtering and normalization on mental health scores, and visualized the distributions and demographic breakdowns.
- The dataset provides valuable insights into psychological symptom measurements and demographic patterns in Kilifi County, serving as a model for AI-ready data standards in Africa.
